# Mistral Pretrained Baseline

This notebook evaluates the pretrained Mistral-7B-Instruct model as a zero-shot abstractive summarizer on the same CNN/DailyMail subset used across the project.

Expected setup:
- GPU runtime recommended
- Kaggle dataset access configured before running download cells
- Hugging Face access may be required depending on runtime and model availability

In [ ]:
!pip install -q transformers datasets peft accelerate bitsandbytes evaluate rouge_score bert_score

from datasets import Dataset
import pandas as pd
import torch
import re
import unicodedata
import evaluate
import os

from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    TrainingArguments, Trainer, BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, TaskType

In [ ]:
# Upload your kaggle.json and enable Kaggle API
from google.colab import files
files.upload()

!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download and unzip dataset
!kaggle datasets download -d gowrishankarp/newspaper-text-summarization-cnn-dailymail --unzip

In [ ]:
# Load 3000-record subset
df = pd.read_csv("cnn_dailymail/train.csv", nrows=3000)
df.columns = [c.strip() for c in df.columns]

# Clean text
def clean_text(text):
    text = unicodedata.normalize("NFKD", text)
    text = re.sub(r"<[^>]+>", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["article"] = df["article"].apply(clean_text)
df["highlights"] = df["highlights"].apply(clean_text)

# Convert to Hugging Face dataset
dataset = Dataset.from_pandas(df)
dataset = dataset.shuffle(seed=42)

train_dataset = dataset.select(range(0, 2400))
val_dataset = dataset.select(range(2400, 2700))
test_dataset = dataset.select(range(2700, 3000))

In [ ]:
from huggingface_hub import login
login()  # This will ask you to paste your token

In [ ]:
model_name = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

In [ ]:
def generate_prompt(article, summary=""):
    return f"""<s>### Instruction:
Summarize the following news article in under 60 words.

### Input:
{article}

### Response:
{summary}</s>"""

In [ ]:
generated_summaries = []
references = []

for i in range(len(test_dataset)):
    example = test_dataset[i]
    prompt = generate_prompt(example["article"])

    with torch.no_grad():
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to("cuda")
        outputs = model.generate(**inputs, max_new_tokens=150)
        summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
        torch.cuda.empty_cache()

    generated_summaries.append(summary)
    references.append(example["highlights"])

    if i % 50 == 0:
        print(f"✅ Completed {i}/{len(test_dataset)}")

In [ ]:
# Load metrics (if not already loaded earlier)
rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")

# Compute ROUGE
rouge_output = rouge.compute(predictions=generated_summaries, references=references)

# Compute BERTScore
bertscore_output = bertscore.compute(predictions=generated_summaries, references=references, lang="en")

# Print Results
print("🔹 ROUGE Scores:")
for k, v in rouge_output.items():
    print(f"{k}: {v:.4f}")

print("\n🔹 BERTScore (F1):")
print(f"F1 Mean: {sum(bertscore_output['f1']) / len(bertscore_output['f1']):.4f}")